# Getting Documents
- Pull the files from the repo location
- Extract the documents we want
- Parse them

In [4]:
!uv add requests

Resolved 119 packages in 879ms                                       
Prepared 1 package in 10ms                                               
Uninstalled 1 package in 3ms
Installed 1 package in 3ms0 (from file:///Users/harunabdulqa
 ~ anvil-llm-lab==0.1.0 (from file:///Users/harunabdulqadir/Documents/GitHub/anvil-llm-lab)


In [5]:
import requests

In [9]:
# Grab the repo with all the documentation (the retrieval location).
# In this case it is a github repo and we will parse it to have it in the right format for the LLM
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [10]:
len(zip_response.content)

17544837

In [11]:
# Turn the file into something the zipfile can read, bytes
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [12]:
# Then we can see what kinds of files we have in there
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [13]:
# Select a file, decode into readable format and take a look at the content
filename = 'docs-main/docs/platform/alerts.mdx'

mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf8')

In [14]:
print(mdx_content)

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:

* A notification channel.

* An alert condition.

## Notification channels

You can choose between the following options:

* **Email**. Add email addresses to send alerts to.

* **Slack**. Add a Slack webhook.

* **Discord**. Add a Discord webhook.

## Alert conditions

### Failed tests

If you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.

<Tip>
  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if th

In [17]:
!uv add python-frontmatter

Resolved 120 packages in 326ms                                       
Prepared 2 packages in 117ms                                             
Uninstalled 1 package in 6ms
Installed 2 packages in 10ms(from file:///Users/harunabdulqa
 ~ anvil-llm-lab==0.1.0 (from file:///Users/harunabdulqadir/Documents/GitHub/anvil-llm-lab)
 + python-frontmatter==1.3.0


In [19]:
# Parse all the files in repo with frontmatter, to read the markdown
import frontmatter

post = frontmatter.loads(mdx_content)

In [23]:
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [20]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [24]:
# Grabbing the correct filename, so we can later refer to the right path
# Example
_, filename_corrected = filename.split('/', maxsplit=1)
print(filename_corrected)

docs/platform/alerts.mdx


In [26]:
# The aim is to turn all the documents into this. Content, title, desc, filename.

doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

print(doc)

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [27]:
# Now to iterate over all the files in the retrieval base
# Loop through the files that we want. Only MDX, MD, not .png or .gif etc

documents = []
with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith(('.md', '.mdx')):
            continue
        with zip_ref.open(file_path) as file:
            content = file.read().decode('utf-8')
            post = frontmatter.loads(content)
            doc = {
                'content': post.content,
                'title': post.metadata.get('title'),
                'description': post.metadata.get('description'),
                'filename': file_path.split('/', 1)[-1]
            }
            documents.append(doc)

In [28]:
len(documents)

95

In [29]:
# A better way to do it is with gitsource packagee from alexey, does everything above simpler.

In [30]:
!uv add gitsource

Resolved 121 packages in 950ms                                       
Prepared 2 packages in 175ms                                             
Uninstalled 1 package in 7ms
Installed 2 packages in 8ms (from file:///Users/harunabdulqa
 ~ anvil-llm-lab==0.1.0 (from file:///Users/harunabdulqadir/Documents/GitHub/anvil-llm-lab)
 + gitsource==0.0.5


In [31]:
# Here you will see we get the same amount of documents
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [32]:
md_file = files[10]

In [33]:
# Have a look at the content
md_file.content

'---\ntitle: \'Output formats\'\ndescription: \'How to export the evaluation results.\'\n---\n\nYou can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python\n

In [34]:
# Parse function
md_file.parse

<bound method RawRepositoryFile.parse of RawRepositoryFile(filename='docs/library/output_formats.mdx', content='---\ntitle: \'Output formats\'\ndescription: \'How to export the evaluation results.\'\n---\n\nYou can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n#

In [35]:
# Parse all the documents
documents = [f.parse() for f in files]

In [36]:
documents[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

# Indexing Data for Search
In RAG we have three steps:

- Search - retrieve relevant documents <-- we are here
- Prompt - prepare the prompt with search results
- LLM - generate the answer


In [38]:
query = 'LLM as a Judge'

In [39]:
!uv add minsearch

Resolved 138 packages in 1.22s                                       
Prepared 11 packages in 1.80s                                            
Uninstalled 1 package in 4ms
Installed 11 packages in 180ms                              
 ~ anvil-llm-lab==0.1.0 (from file:///Users/harunabdulqadir/Documents/GitHub/anvil-llm-lab)
 + cloudpickle==3.1.2
 + joblib==1.6.0
 + minsearch==0.2.0
 + numpy==2.2.6
 + pandas==2.3.3
 + pytz==2026.5
 + scikit-learn==1.7.2
 + scipy==1.15.3
 + stemlite==0.1.0
 + threadpoolctl==3.7.0


In [ ]:
from minsearch import Index

In [ ]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [ ]:
results = index.search(query, num_results=5)

In [49]:
len(results)

5

In [48]:
results[4]

{'title': 'LLM Evaluation',
 'description': 'Evaluate text outputs in under 5 minutes',
 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nEvidently helps you evaluate LLM outputs automatically. The lets you compare prompts, models, run regression or adversarial tests with clear, repeatable checks. That means faster iterations, more confident decisions, and fewer surprises in production.\n\nIn this Quickstart, you\'ll try a simple eval in Python and view the results in Jupyter notebook or Colab. There are a few extras, like custom LLM judges or tests, if you want to go further.\n\nLet’s dive in.\n\n<Info>\n  Need help at any point? Ask on [Discord](https://discord.com/invite/xZjKRaNp8b).\n</Info>\n\n## 1. Set up your environment\n\nInstall the Evidently Python library:\n\n```python\n!pip install evidently\n```\n\nComponents to run the evals:\n\n```python\nimport pandas as pd\nfrom evidently import Dataset

# Chunking
- Long documents cause trouble at the search step. If we retrieve 5 documents and each is 20,000 characters, that's 100,000 characters going to the LLM. We're paying for all those tokens, and it may overwhelm the LLM or make it less effective.

- We need to process documents into smaller, focused pieces.

Example:
- search <-- 5 docs
- prompt <-- 5 x 20k = 100k
- llm

In [50]:
# Have a look at the size of the troublesome files

doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [51]:
document = list(range(0, 100))

In [53]:
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [54]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})
        
        start = end - step
        
        if end >= text_length:
            break
    
    return chunks

In [55]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [57]:
documents[26]

# we only want to chunk up the content as you can see there, we dont need to touch the title, descrip, or filename

{'title': 'Explore view',
 'description': 'Reviewing the evaluation results on the Platform.',
 'content': 'The result of each evaluation is a Report (summary of metrics with visuals) with an optional Test Suite (when it also includes pass/fail results on set conditions).\n\n**Browse the results**. To access the results of your evaluations, enter your Project and navigate to the "Reports" section in the left menu. Here, you can view all your evaluation artifacts and browse them by Tags, time, or metadata. You can also download them as HTML or JSON.\n\n![](/images/evals_browse_reports-min.png)\n\nTo see and compare the evaluation results, click on "Explore" next to the individual Report.\n\n**Explore view**. You\'ll get the Report or Test Suite and, if available, the dataset linked to the evaluation.\n\n![](/images/evals_explore_view-min.png)\n\n- To view the Report only, click on the "Dataset" sign at the top to hide the dataset.\n- To explore the Dataset only, choose "Go to dataset".\

In [58]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)
    
    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [59]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [60]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [61]:
results = chunk_index.search(query)

In [63]:
from gitsource import chunk_documents

In [64]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [66]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

# RAG

In [68]:
!uv add openai

Resolved 138 packages in 6ms
Checked 124 packages in 119ms


In [72]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [73]:
search_result = chunk_index.search(query, num_results=5)

In [74]:
search_result

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont